# Magnitudes from photon counting devices

The definition of a magnitude is given by 

\begin{equation}
m_1 - m_2 = -2.5 {\rm log_{10}} \left( F_1 / F_2 \right)
\end{equation}

where $F_1$ and $F_2$ are respectively the fluxes of objects 1 and 2 in the band. See Equation 5.2 in Rieke. 

As discussed previously, for a Vega-based magnitude system (i.e. where Vega is magnitude zero by defintion) then  

\begin{equation}
m_1 - 0 = -2.5 {\rm log} \left( \frac{ \int f_{\lambda,1} \epsilon(\lambda) d\lambda } { \int f_{\lambda,Vega} \epsilon(\lambda) d\lambda } \right)
\end{equation}

where $f_{\lambda,1}$ and $f_{\lambda,Vega}$ and $\epsilon(\lambda)$ is our quantum efficiency (telescope+filter+CCD) as a function of wavelength.

When we use a photon counting device like a CCD, then the magnitudes we measure with such devices are given by

\begin{equation}
m_1 - 0 = -2.5 {\rm log} \left( \frac{ \int f_{\lambda,1} \epsilon(\lambda) / (h\nu) d\lambda } { \int f_{\lambda,Vega} \epsilon(\lambda) / (h\nu) d\lambda } \right)
\end{equation}

which is dividing the spectrum by the energy per photon. This can be rewritten as

\begin{equation}
m_1 - 0 = -2.5 {\rm log} \left( \frac{ \int f_{\lambda,1} \epsilon(\lambda) \lambda d\lambda } { \int f_{\lambda,Vega} \epsilon(\lambda) \lambda d\lambda } \right)
\end{equation}


where $f_{\lambda,1}$ and $f_{\lambda,Vega}$ and $\epsilon(\lambda)$ is our quantum efficiency (telescope+filter+CCD) as a function of wavelength.

In practice this changes the magnitudes very little. 

In [ ]:
import numpy as np
from astropy.io import ascii
from astropy.coordinates import SkyCoord
from astropy import units as u
from astroquery.gaia import Gaia
import matplotlib.pyplot as plt


In [ ]:
# Read a two column ascii file - only very slightly modified from what you can find on the internet

def Read_Two_Column_File(file_name):
    with open(file_name, 'r') as data:
        x = []   # Define an array with no elements
        y = []   # Define an array with no elements
        for line in data:
            if (line[0]!='#'):          # If the line is uncommented by a hash then
                p = line.split()        # Split the line components (i.e., the columns)
                x.append(float(p[0]))   # Add a new element to x
                y.append(float(p[1]))   # Add a new element to y

    return x, y

In [ ]:
# Produce an interpolated flam (or transmission) - note this is a simple but very inefficient implementation

def Flam_interpolate(lam, star_lam, star_flam):
    flam = 0
    i=0
    if (lam>star_lam[0]) and (lam<star_lam[-1]):
        while star_lam[i]<lam:
            i = i + 1
        flam = (star_flam[i-1]*(star_lam[i]-lam) + star_flam[i]*(lam-star_lam[i-1])) / (star_lam[i]-star_lam[i-1])   
    return flam

In [ ]:
def Flam_to_magnitude(star_lam, star_flam, vega_lam, vega_flam, fil_lam, fil_tran):

    # Initial values
    top=0.0
    bottom=0.0
    lam=fil_lam[0]
    dlam=1.0
    mag=-99.9

    # Step through lamba range of the filter
    while (lam<fil_lam[-1]):

        # Top line - star flux per unit lambda multiplied by filter transmisison and dlambda
        top = top + Flam_interpolate(lam,star_lam,star_flam) * Flam_interpolate(lam,fil_lam,fil_tran) * dlam
        
        # Bottom line - Vega flux per unit lamdba multiplied by filter transmisson and dlambda
        bottom = bottom + Flam_interpolate(lam,vega_lam,vega_flam) * Flam_interpolate(lam,fil_lam,fil_tran) * dlam
        
        # Increment lambda by delta lambda 
        lam = lam + dlam

    # If integrals have positive values return a magnitude that isn't -99
    if (top>0.0) and (bottom>0.0):
         mag = -2.5 * np.log10( top / bottom)

    # Return the magnitude value        
    return mag    


In [ ]:
def Flam_to_ccdmagnitude(star_lam, star_flam, vega_lam, vega_flam, fil_lam, fil_tran):

    # Initial values
    top=0.0
    bottom=0.0
    lam=fil_lam[0]
    dlam=1.0
    mag=-99.9

    # Step through lamba range of the filter
    while (lam<fil_lam[-1]):

        # Top line - star flux per unit lambda multiplied by filter transmisison, lambda and dlambda
        top = top + Flam_interpolate(lam,star_lam,star_flam) * Flam_interpolate(lam,fil_lam,fil_tran) * lam * dlam
        
        # Bottom line - Vega flux per unit lamdba multiplied by filter transmisson, lambada and dlambda
        bottom = bottom + Flam_interpolate(lam,vega_lam,vega_flam) * Flam_interpolate(lam,fil_lam,fil_tran) * lam * dlam
        
        # Increment lambda by delta lambda 
        lam = lam + dlam

    # If integrals have positive values return a magnitude that isn't -99
    if (top>0.0) and (bottom>0.0):
         mag = -2.5 * np.log10( top / bottom)

    # Return the magnitude value        
    return mag    


In [ ]:
#  Vega ftp://ftp.eso.org/pub/stecf/standards/hststan/fhr7001.dat
vega_lam, vega_flam = Read_Two_Column_File('fhr7001.dat')    # Vega units Ang and ergs/cm/cm/s/A * 10**16
print(vega_lam[0], vega_flam[0])
print(vega_lam[1], vega_flam[1])
print(vega_lam[-1], vega_flam[-1])

In [ ]:
# Star ftp://ftp.eso.org/pub/stecf/standards/hststan/
star_lam, star_flam = Read_Two_Column_File('ffeige34.dat')   # Star units Ang and erg/cm/cm/s/A * 10**16
print(star_lam[0], star_flam[0])
print(star_lam[1], star_flam[2])
print(star_lam[-1], star_flam[-1])

In [ ]:
# Filter  V_bessell.dat 
fil_lam, fil_tran = Read_Two_Column_File('V_bessell.dat')  
print(fil_lam[0], fil_tran[0])
print(fil_lam[1], fil_tran[1])
print(fil_lam[-1], fil_tran[-1])
print(4750, Flam_interpolate(4750, fil_lam, fil_tran))

In [ ]:
# Lets run function. Please note the wavelengths and flam need consistent units. Tranmission can be fractional or percentage.

mag=Flam_to_magnitude(star_lam, star_flam, vega_lam, vega_flam, fil_lam, fil_tran)
print('Magnitude:', mag)

# Lets run function. Please note the wavelengths and flam need consistent units. Tranmission can be fractional or percentage.

ccdmag=Flam_to_ccdmagnitude(star_lam, star_flam, vega_lam, vega_flam, fil_lam, fil_tran)
print('CCD Magnitude:', ccdmag)

# Zeropoint 

From a practical standpoint, to calibrate a CCD images, we are not going to observe stars with precision spectra and then synthesise magnitudes in our filters. 

Instead we are going to observe stars with known magnitudes in our filters or very similar filters. In that case we can use 

\begin{equation}
m_1 - m_2 = -2.5 {\rm log_{10}} \left( F_1 / F_2 \right)
\end{equation}

where $m_2$ is known and $F_1$ and $F_2$ can be measured off the images in counts per second. If have a star of known magnitude in your science image, then this is trivial. That said multiple stars with known magnitudes can improve the calibration and can be used to avoid errors. 

It can be useful to define a zero point where 

\begin{equation}
m_1 - 0 = -2.5 {\rm log_{10}} \left( F_1 / F_{zp} \right)
\end{equation}

where $F_1$ are the counts per second from an object of interest and $F_{zp}$ is the counts per second that would be produced by a magnitude zero star (Equation 5.3 from Rieke). This can also be rewritten as 

\begin{equation}
m_1 - 0 = zp -2.5 {\rm log_{10}} \left( F_1  \right)
\end{equation}

where $F_1$ are the counts per second from an object of interest and $zp$ is the magnitude of a star that would produce 1 count per second.  

Rather than observing a magnitude zero star (too bright) or a star producing one count per second (too faint), we observe "standard" stars of known magnitude so

\begin{equation}
m_1 - m_{standard} = -2.5 {\rm log_{10}} \left( \frac{F_1}{F_{standard}}  \right)
\end{equation}

(Equation 5.4 from Rieke) where $F_{standard}$ are the counts per second from the standard star. In this case 

\begin{equation}
m_1 - 0 = zp -2.5 {\rm log_{10}} \left( F_1  \right)
\end{equation}

leads to 

\begin{equation}
zp = m_{standard} + 2.5 {\rm log_{10}} \left( F_{standard}  \right)
\end{equation}

So, for your science projects, if you can identify multiple stars of known magnitude in your science images, you can determine a zeropoint. 

That zeropoint can then be used to determine photometry for all the other objects in your science image.

Since the flux and magnitudes will depend on the filter you're using, a zeropoint must be determined for every filter. 

# More general photometric calibration

For a perfectly clear night - a photometric night - you can define a more general photometric solution that can be used for all images, irrespsective of whether those images are have stars of known magnitudes in them.

In this case 

\begin{equation}
m_{1,1} - 0 = zp -2.5 {\rm log_{10}} \left( F_1  \right) - A\kappa + C (m_{1,1} - m_{1,2}) 
\end{equation}

where $A$ is the airmass, $m_{1,1}$ is the magnitude of the object in filter 1 and $m_{1,2}$ is the magnitude of the object in filter 2. $\kappa$ and $C$ are constants determined for the relevant photometric night that account for the transmission of light through the atmosphere and the difference between our filters and the "standard" filters.


# Atmosphere and airmass

The atmosphere absorbs and scatters the light that passes through it, with Rayleigh scattering being the most significant issue in visible wavelengths. 

The amount of absorption and scattering depends on the amount of air the light passes through. If 80\% of light is transmitted through one unit of astmosphere then 64\% of light is transmitted through two units of atmosphere (i.e. $0.8^2$) 

We quantity the amount of air light has to pass through with the airmass, which is defined as one for the amount of air directly overhead (i.e. zenith). 

The airmass is then approximately

\begin{equation}
A \simeq \frac{1}{{\rm cos} (z)} = {\rm sec} (z) 
\end{equation}

where $z$ is the angle from zenith. More precisely it is given by Equation 5.8 of Rieke, which accounts for refraction and the curvature of the Earth

\begin{equation}
A  = {\rm sec} (z) - 0.0018167 ({\rm sec} (z) - 1) - 0.002875 ({\rm sec} (z) - 1)^2 - 0.0008083  ({\rm sec} (z) - 1)^3
\end{equation}

The magnitude system works in our favour, so the change in magnitude as a function of airmass is just $\kappa A$. 

As $\kappa$ will vary from location and location and night to night (even on clear nights), it needs to be determined each clear night using standards observed at different zenith angles. Typically $\kappa$ is somewhere between 0.1 and 0.5 magnitudes per unit airmass depending on the band and site. 


# Colour term

Unfortunately any telescope, filter and detector combination we use will never quite match the standard filter used to define the relevant magnitudes (e.g. UBVRI, ugriz, JHK).

The difference between magnitudes measure in two filters will depend on the shape of the filters and the shape of the spectrum of the object. However, if the filters are near each other the difference as a function of colour (i.e. spectrum) can be approximated by a Taylor expansion. This is the last term in 

\begin{equation}
m_{1,1} - 0 = zp -2.5 {\rm log_{10}} \left( F_1  \right) - A\kappa + C (m_{1,1} - m_{1,2}) 
\end{equation}

We thus need to determine $C$ by observing many stars with known magnitudes and colours on a perfectly clear night. 

$C$ will obviously depend on the filters being used, but on the plus side it should be stable from night to night (i.e. your filters shouldn't be changing their transmission). If solutions to the equation above (a photometric solution) determined on two nights have very different values of $C$, this may indicate one of the nights was not photometric. 

<IMG SRC="ExampleB.jpg" width=700>

# Photometric calibration from archival data 

When stars of well determined magnitude were sparsely scattered across the sky, the photometric solution discussed above was required. 

Today there are large numbers of stars with known magnitudes observed across the entire sky. 

However, these stars are not necessarily observed in the same filters that we are using, so we need to synthesise magnitudes. 

For example, the Gaia satellite uses its own blue, green and red bands whereas our filters resember the Johnson-Cousins UBVRI filters.

UBVRI corresponds to ultraviolet, blue, visible, red and infrared (although that infrared is only 0.8 micron).

We need to determine the relationship between Johnson-Cousins UBVRI and Gaia photometry.


# Loading UBVRI photometry from a CSV files as a Table 

In [ ]:
# Lets load a dataset of known UBVRI magnitudes from Graham 1982

Graham = ascii.read("Graham1982.csv", format='csv', fast_reader=False)


In [ ]:
# Lets print the data 
print('The table') 
print(Graham) 
print('\n\n') 

# Lets print the first line
print('The first line') 
print(Graham[0])
print('\n\n') 

# Lets print a column
print('Print a Column') 
print(Graham['Vmag'])
print('\n\n') 

# Print the length of the table 
print('Length:', len(Graham))
print('\n\n') 


# Matching to GAIA 

Lets match this data to GAIA

Gaia has blue, (broad) green and red bands we will call GBP, G and GRP.

Lets add these columns to the table, with rubbish values of -99.9



In [ ]:
Graham.add_column(-99.9, name='GBP') 
Graham.add_column(-99.9, name='G') 
Graham.add_column(-99.9, name='GRP') 


In [ ]:
print(Graham)

In [ ]:
# Query the Gaia 

# Lets start with the first star
Star=Graham[0]
print(Star)

# Convert coordinates to decimal degrees 
rad=15.0*(float(Star['RAh'])+float(Star['RAm'])/60.0+float(Star['RAs'])/3600.0)
if '-' in Star['DE-']:
    decd=0.0-float(Star['DEd'])-float(Star['DEm'])/60.0-float(Star['DEs'])/3600.0
else:
    decd=float(Star['DEd'])+float(Star['DEm'])/60.0+float(Star['DEs'])/3600.0
print(rad, decd)

# Get coordinates in astropy format
coord = SkyCoord(ra=rad, dec=decd, unit=(u.degree, u.degree), frame='icrs')
# Match radius in degrees
radius = u.Quantity(0.001, u.deg)
# Run query and return results
j = Gaia.cone_search_async(coord, radius)
r = j.get_results()
# If there is a result then print the first match
if len(r)>0:
    Star['GBP']=r[0]['phot_bp_mean_mag']
    Star['G']=r[0]['phot_g_mean_mag']
    Star['GRP']=r[0]['phot_rp_mean_mag']
print('\n')
print('V    B-V    GBP                 G             GRP')
print(Star['Vmag'], Star['B-V'], Star['GBP'], Star['G'], Star['GRP'])
print('\n\n')

# Find GAIA photometry for as many Graham stars as possible

In [ ]:
for Star in Graham:
    print(Star)
    
    # Convert coordinates to decimal degrees 
    rad=15.0*(float(Star['RAh'])+float(Star['RAm'])/60.0+float(Star['RAs'])/3600.0)
    if '-' in Star['DE-']:
        decd=0.0-float(Star['DEd'])-float(Star['DEm'])/60.0-float(Star['DEs'])/3600.0
    else:
        decd=float(Star['DEd'])+float(Star['DEm'])/60.0+float(Star['DEs'])/3600.0
    print(rad, decd)

    # Get coordinates in astropy format
    coord = SkyCoord(ra=rad, dec=decd, unit=(u.degree, u.degree), frame='icrs')
    # Match radius in degrees
    radius = u.Quantity(0.001, u.deg)
    # Run query and return results
    j = Gaia.cone_search_async(coord, radius)
    r = j.get_results()
    # If there is a result then print the first match
    if len(r)>0:
        Star['GBP']=r[0]['phot_bp_mean_mag']
        Star['G']=r[0]['phot_g_mean_mag']
        Star['GRP']=r[0]['phot_rp_mean_mag']
    print('\n')
    print('V    B-V    GBP                 G             GRP')
    print(Star['Vmag'], Star['B-V'], Star['GBP'], Star['G'], Star['GRP'])
    print('\n\n')

# Plot V - G as a function of GBP - G

In [ ]:
# Plot the difference between V and G as a function of GAIA G_BP and G.

# What is px and py? What is the if statement doing?
px=[]        # Comment
py=[]        # Comment
for Star in Graham:
    # Check if we have valid Graham and GAIA magnitudes and colours.
    # Remember we sometimes use -99 to denote a missing value. 
    if Star['Vmag']>0.0 and Star['Vmag']<30.0 and \
    Star['GBP']>0.0 and Star['GBP']<30.0 and \
    Star['G']>0.0 and Star['G']<30.0 and \
    (Star['Vmag']-Star['G'])**2<2.25 :         # Only match if V and G are similar
        px.append(Star['GBP']-Star['G'])      # G_BP minus G
        py.append(Star['Vmag']-Star['G'])          # V minus G

plt.title('V minus G as a function of colour')       # Title
plt.ylabel('Graham V - GAIA G')                      # y-label
plt.xlabel('GAIA G_BP - GAIA G')                     # x-label
plt.scatter(px, py, marker='o');                     # What is this plotting?    


In [ ]:
# Fit to this data 

# What is px and py? What is the if statement doing?
px=[]        # Comment
py=[]        # Comment
for Star in Graham:
    # Check if we have valid Graham and GAIA magnitudes and colours.
    # Remember we sometimes use -99 to denote a missing value. 
    if Star['Vmag']>0.0 and Star['Vmag']<30.0 and \
    Star['GBP']>0.0 and Star['GBP']<30.0 and \
    Star['G']>0.0 and Star['G']<30.0 and \
    Star['GBP']-Star['G']>0.0 and \
    (Star['Vmag']-Star['G'])**2<2.25 :         # Only match if V and G are similar
        px.append(Star['GBP']-Star['G'])      # G_BP minus G
        py.append(Star['Vmag']-Star['G'])          # V minus G

# What does polyfit do doing?
Vpoly=np.polyfit(px, py, 2)                           
print(Vpoly)                                          

# What are plx and ply? If you are unsure print their values.
plx=np.arange(-0.1, 1.3, 0.1).tolist()
ply=[]
for x in plx:
    ply.append(Vpoly[2]+Vpoly[1]*x+Vpoly[0]*x*x)

plt.title('V minus G as a function of colour')       # Title
plt.ylabel('Graham V - GAIA G')                      # y-label
plt.xlabel('GAIA G_BP - GAIA G')                     # x-label
plt.scatter(px, py, marker='o');                     # What is this plotting?    
plt.plot(plx, ply, linestyle='dashed', linewidth=2); # What is this plotting?


So in this instance 

\begin{equation}
V-G = f(BP-G)
\end{equation}

or 

\begin{equation}
V = G + f(BP-G)
\end{equation}

#### From the polynomial fit [0.4438681  0.26560882 0.0039465 ]

\begin{equation}
V = G + 0.4438681 \times (BP-G)^2 + 0.26560882 \times (BP-G) + 0.0039465
\end{equation}


In [ ]:
Graham.write('GrahamGaia.dat', format='csv', overwrite=True)  

# Observing night 

The calibrations we require define our observing nights

We need to arrive before sunset and cool the camera (to lower dark current) 

We need to take bias and dark exposures 

We open the dome and uncover the telescope, taking exposures of the twilight sky (with tracking off)

We move to bright stars and focus the telescope (with tracking on)

If it is clear and if they are needed, we observe standard stars to get the zeropoint, airmass and colour terms

We take our science exposures

If it is clear and if they are needed, we again observe standard stars to get the zeropoint, airmass and colour terms

Park the telescope and close the dome

We need to take bias and dark exposures if needed
